<a href="https://colab.research.google.com/github/SasankaPandaSCIT/Automate-with-Gen-AI-Agents/blob/main/Module%205/5.3%20Task%20Flow%2C%20Memory%2C%20and%20Evaluation/5.3.1%20Tutorial_%20Building%20a%20Multi-Tool%20Agent%20with%20Task%20Memory%20Openrouter.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Tutorial: Building a Multi-Tool Agent with Task Memory


In this tutorial, we'll build an agent that can use multiple tools AND remember
what it learned from previous tasks. Think of it like giving your agent a toolbox
and a notebook to write down what works.

Learning Objectives:
- Create agents with multiple specialized tools
- Add simple memory to remember past successes
- Build agents that get better over time
- See how memory improves agent performance

In [1]:
# Install required packages
!pip install -q "pydantic-ai-slim[openrouter]==2.48.0" openai==3.19.0 pydantic==2.13.5 python-dotenv==1.2.3

import os
from typing import List, Dict, Any
from datetime import datetime
from pydantic import BaseModel, Field
from pydantic_ai import Agent, RunContext
from dotenv import load_dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.5/125.5 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.4/169.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.6/146.6 kB 6.1 MB/s eta 0:00:00


In [2]:
# Load environment variables
load_dotenv()

False

In [4]:
# OpenRouter setup
from getpass import getpass
from pydantic_ai.models.openrouter import OpenRouterModel
from pydantic_ai.providers.openrouter import OpenRouterProvider

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY") or getpass("Enter your OpenRouter API key: ")
MODEL = "openai/gpt-4.1-mini"

# PydanticAI model routed through OpenRouter; used by every agent below.
openrouter_model = OpenRouterModel(MODEL, provider=OpenRouterProvider(api_key=OPENROUTER_API_KEY))

Enter your OpenRouter API key: ··········


### Simple Memory System

Let's start with a basic memory system to store what our agent learns.


In [5]:
class TaskMemory(BaseModel):
    """
    Simple memory of a completed task.
    """
    task_type: str = Field(description="What kind of task this was")
    what_worked: str = Field(description="What approach worked well")
    result_quality: float = Field(description="How good the result was (0-1)")
    timestamp: datetime = Field(description="When this was completed")


In [6]:
class SimpleMemoryBank:
    """
    A simple memory bank for our agent.
    """

    def __init__(self):
        self.memories: List[TaskMemory] = []

    def remember_task(self, task_type: str, what_worked: str, quality: float):
        """Remember what worked for a task."""
        memory = TaskMemory(
            task_type=task_type,
            what_worked=what_worked,
            result_quality=quality,
            timestamp=datetime.now()
        )
        self.memories.append(memory)
        print(f"💾 Remembered: {what_worked}")

    def recall_experience(self, task_type: str) -> List[TaskMemory]:
        """Recall past experience with similar tasks."""
        return [m for m in self.memories if task_type.lower() in m.task_type.lower()]


### Creating Multiple Tools

Now let's give our agent several different tools to work with.


In [7]:
# Initialize our memory bank
memory_bank = SimpleMemoryBank()

class MultiToolResponse(BaseModel):
    """
    Response from our multi-tool agent.
    """
    answer: str = Field(description="The main answer")
    tools_used: List[str] = Field(description="Which tools were used")
    lessons_learned: List[str] = Field(description="What the agent learned")
    improved_from_memory: bool = Field(description="Whether past experience helped")


In [8]:
multi_tool_agent = Agent(
    openrouter_model,
    output_type=MultiToolResponse,
    system_prompt="""
    You are a multi-tool agent with memory capabilities.

    Your approach:
    1. Check if you have experience with similar tasks
    2. Use the most appropriate tools for the job
    3. Learn from the results and remember what works
    4. Get better at tasks over time

    Always explain which tools you used and what you learned.
    """
)

In [9]:
@multi_tool_agent.tool
async def research_tool(ctx: RunContext[None], topic: str) -> str:
    """
    Research information about a topic.
    """
    # Check if we have experience with this type of research
    past_experience = memory_bank.recall_experience("research")

    experience_note = ""
    if past_experience:
        best_approach = max(past_experience, key=lambda x: x.result_quality)
        experience_note = f"Using past experience: {best_approach.what_worked}"

    # Simulated research results
    result = f"""
    Research on: {topic}
    {experience_note}

    Key findings:
    - Main concept: {topic} is an important subject with multiple aspects
    - Current trends: Growing interest and development
    - Applications: Various practical uses across industries

    Research approach: Web search + expert analysis
    """

    # Remember what worked
    memory_bank.remember_task("research", "comprehensive web search with expert analysis", 0.8)

    return result

In [10]:
@multi_tool_agent.tool
async def analysis_tool(ctx: RunContext[None], data: str, analysis_type: str) -> str:
    """
    Analyze data using different approaches.
    """
    # Check past analysis experience
    past_experience = memory_bank.recall_experience("analysis")

    improvement = ""
    if past_experience:
        improvement = "Applied lessons from previous analyses"

    # Perform analysis
    result = f"""
    Analysis of: {data[:50]}...
    Type: {analysis_type}
    {improvement}

    Analysis Results:
    - Pattern identified: Clear trends in the data
    - Key insights: Multiple important findings
    - Recommendations: 3 actionable next steps

    Confidence: High
    """

    # Remember this approach
    memory_bank.remember_task("analysis", f"{analysis_type} analysis with pattern recognition", 0.9)

    return result


In [11]:
@multi_tool_agent.tool
async def planning_tool(ctx: RunContext[None], goal: str, constraints: List[str]) -> str:
    """
    Create plans considering goals and constraints.
    """
    # Use planning experience
    past_experience = memory_bank.recall_experience("planning")

    experience_boost = ""
    if past_experience:
        experience_boost = "Leveraging past planning successes"

    result = f"""
    Plan for: {goal}
    {experience_boost}

    Constraints considered: {len(constraints)} factors

    Recommended Plan:
    1. Phase 1: Foundation setup (Week 1-2)
    2. Phase 2: Core implementation (Week 3-4)
    3. Phase 3: Testing and refinement (Week 5-6)

    Success factors: Clear milestones, regular reviews
    """

    # Store planning wisdom
    memory_bank.remember_task("planning", "phased approach with clear milestones", 0.85)

    return result

### Memory-Enhanced Decision Making

Let's add a tool that helps the agent decide which approach to use based on memory.


In [12]:
@multi_tool_agent.tool
async def check_memory_for_guidance(ctx: RunContext[None], task_type: str) -> str:
    """
    Check memory to guide decision making.
    """
    relevant_memories = memory_bank.recall_experience(task_type)

    if not relevant_memories:
        return f"No past experience with {task_type} tasks. Will use standard approach."

    # Find the most successful approach
    best_memory = max(relevant_memories, key=lambda x: x.result_quality)

    guidance = f"""
    Memory guidance for {task_type}:

    Past attempts: {len(relevant_memories)}
    Best approach: {best_memory.what_worked}
    Success rate: {best_memory.result_quality:.1%}

    Recommendation: Use proven successful approach with minor improvements.
    """

    return guidance

In [13]:
print("=== TASK 1: First Research (No Memory) ===")
result1 = await multi_tool_agent.run("Research artificial intelligence applications")
print(f"Tools used: {result1.output.tools_used}")
print(f"Memory helped: {result1.output.improved_from_memory}")
print(f"Lessons: {result1.output.lessons_learned}")
print()

# Task 2: Second research task (with memory)
print("=== TASK 2: Second Research (With Memory) ===")
result2 = await multi_tool_agent.run("Research machine learning trends")
print(f"Tools used: {result2.output.tools_used}")
print(f"Memory helped: {result2.output.improved_from_memory}")
print(f"Lessons: {result2.output.lessons_learned}")
print()

# Task 3: Planning task
print("=== TASK 3: Planning Task ===")
result3 = await multi_tool_agent.run("Plan a product launch strategy")
print(f"Tools used: {result3.output.tools_used}")
print(f"Answer preview: {result3.output.answer[:100]}...")
print()

=== TASK 1: First Research (No Memory) ===
💾 Remembered: comprehensive web search with expert analysis
Tools used: ['functions.research_tool', 'functions.check_memory_for_guidance']
Memory helped: False
Lessons: ['AI applications are diverse and span many industries.', 'Research benefits from combining web search data with expert analysis.']

=== TASK 2: Second Research (With Memory) ===
💾 Remembered: comprehensive web search with expert analysis
Tools used: ['research_tool', 'check_memory_for_guidance']
Memory helped: False
Lessons: ['Using web search combined with expert analysis provided comprehensive and up-to-date insights.', 'No prior similar research experience was found, confirming the need to rely on a broad research approach.']

=== TASK 3: Planning Task ===
💾 Remembered: phased approach with clear milestones
💾 Remembered: phased approach with clear milestones
Tools used: ['check_memory_for_guidance', 'planning_tool']
Answer preview: A comprehensive product launch strategy is

In [14]:
print("=== MEMORY BANK SUMMARY ===")
print(f"Total memories: {len(memory_bank.memories)}")
for memory in memory_bank.memories:
    print(f"- {memory.task_type}: {memory.what_worked} (Quality: {memory.result_quality:.1f})")


=== MEMORY BANK SUMMARY ===
Total memories: 4
- research: comprehensive web search with expert analysis (Quality: 0.8)
- research: comprehensive web search with expert analysis (Quality: 0.8)
- planning: phased approach with clear milestones (Quality: 0.8)
- planning: phased approach with clear milestones (Quality: 0.8)
